In [0]:
# QuickBite Streaming Setup

STREAM_SOURCE = "/Volumes/quickbite/default/raw_files/streaming_events"
CHECKPOINT_PATH = "/Volumes/quickbite/default/raw_files/checkpoints/quickbite_order_events"
BRONZE_TABLE = "quickbite.default.bronze_order_events"

dbutils.fs.mkdirs(STREAM_SOURCE)
dbutils.fs.mkdirs(CHECKPOINT_PATH)

print("Streaming source:", STREAM_SOURCE)
print("Checkpoint:", CHECKPOINT_PATH)
print("Bronze table:", BRONZE_TABLE)

Streaming source: /Volumes/quickbite/default/raw_files/streaming_events
Checkpoint: /Volumes/quickbite/default/raw_files/checkpoints/quickbite_order_events
Bronze table: quickbite.default.bronze_order_events


In [0]:
# QuickBite - Streaming Drop 01

drop_01 = """
{"event_id":"EVT000001","order_id":"QBO10001","event_timestamp":"2026-10-03T18:00:00","event_type":"ORDER_PLACED","city":"Hyderabad","restaurant_id":"QBR0308","rider_id":null,"order_value_inr":425.50,"payment_method":"UPI","source_system":"QUICKBITE_LIVE"}
{"event_id":"EVT000002","order_id":"QBO10001","event_timestamp":"2026-10-03T18:02:00","event_type":"RESTAURANT_ACCEPTED","city":"Hyderabad","restaurant_id":"QBR0308","rider_id":null,"order_value_inr":425.50,"payment_method":"UPI","source_system":"QUICKBITE_LIVE"}
{"event_id":"EVT000003","order_id":"QBO10001","event_timestamp":"2026-10-03T18:05:00","event_type":"RIDER_ASSIGNED","city":"Hyderabad","restaurant_id":"QBR0308","rider_id":"QBD01314","order_value_inr":425.50,"payment_method":"UPI","source_system":"QUICKBITE_LIVE"}
{"event_id":"EVT000004","order_id":"QBO10002","event_timestamp":"2026-10-03T18:06:00","event_type":"ORDER_PLACED","city":"Pune","restaurant_id":"QBR0511","rider_id":null,"order_value_inr":315.75,"payment_method":"CARD","source_system":"QUICKBITE_LIVE"}
"""

dbutils.fs.put(
    f"{STREAM_SOURCE}/drop_01.json",
    drop_01,
    overwrite=True
)

print("Drop 01 created successfully.")
display(dbutils.fs.ls(STREAM_SOURCE))

Wrote 1040 bytes.
Drop 01 created successfully.


path,name,size,modificationTime
dbfs:/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json,drop_01.json,1040,1791042972000


In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

# Paths
SCHEMA_PATH = "/Volumes/quickbite/default/raw_files/checkpoints/quickbite_order_events/schema"
QUERY_CHECKPOINT = "/Volumes/quickbite/default/raw_files/checkpoints/quickbite_order_events/query"

# Explicit schema for live order events
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("event_timestamp", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("rider_id", StringType(), True),
    StructField("order_value_inr", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("source_system", StringType(), True)
])

# Auto Loader
stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .schema(event_schema)
        .load(STREAM_SOURCE)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.input_file_name())
)

print("Auto Loader streaming DataFrame created successfully.")

Auto Loader streaming DataFrame created successfully.


In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

# Paths
SCHEMA_PATH = "/Volumes/quickbite/default/raw_files/checkpoints/quickbite_order_events/schema"
QUERY_CHECKPOINT = "/Volumes/quickbite/default/raw_files/checkpoints/quickbite_order_events/query"

# Explicit schema
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("event_timestamp", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("rider_id", StringType(), True),
    StructField("order_value_inr", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("source_system", StringType(), True)
])

# Auto Loader
stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .schema(event_schema)
        .load(STREAM_SOURCE)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
)

print("Auto Loader streaming DataFrame created successfully.")

Auto Loader streaming DataFrame created successfully.


In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", QUERY_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print("Streaming Bronze ingestion completed successfully.")

Streaming Bronze ingestion completed successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {BRONZE_TABLE}
        ORDER BY event_timestamp
    """)
)

event_id,order_id,event_timestamp,event_type,city,restaurant_id,rider_id,order_value_inr,payment_method,source_system,_ingested_at,_source_file
EVT000001,QBO10001,2026-10-03T18:00:00.000Z,ORDER_PLACED,Hyderabad,QBR0308,null,425.5,UPI,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000002,QBO10001,2026-10-03T18:02:00.000Z,RESTAURANT_ACCEPTED,Hyderabad,QBR0308,null,425.5,UPI,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000003,QBO10001,2026-10-03T18:05:00.000Z,RIDER_ASSIGNED,Hyderabad,QBR0308,QBD01314,425.5,UPI,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000004,QBO10002,2026-10-03T18:06:00.000Z,ORDER_PLACED,Pune,QBR0511,null,315.75,CARD,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json


In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS bronze_event_count
    FROM {BRONZE_TABLE}
""").show()

+------------------+
|bronze_event_count|
+------------------+
|                 4|
+------------------+



In [0]:
# QuickBite - Streaming Drop 02

drop_02 = """
{"event_id":"EVT000005","order_id":"QBO10002","event_timestamp":"2026-10-03T18:08:00","event_type":"RESTAURANT_ACCEPTED","city":"Pune","restaurant_id":"QBR0511","rider_id":null,"order_value_inr":315.75,"payment_method":"CARD","source_system":"QUICKBITE_LIVE"}
{"event_id":"EVT000006","order_id":"QBO10002","event_timestamp":"2026-10-03T18:11:00","event_type":"RIDER_ASSIGNED","city":"Pune","restaurant_id":"QBR0511","rider_id":"QBD02107","order_value_inr":315.75,"payment_method":"CARD","source_system":"QUICKBITE_LIVE"}
{"event_id":"EVT000007","order_id":"QBO10003","event_timestamp":"2026-10-03T18:12:00","event_type":"ORDER_PLACED","city":"Bengaluru","restaurant_id":"QBR0724","rider_id":null,"order_value_inr":589.00,"payment_method":"UPI","source_system":"QUICKBITE_LIVE"}
"""

dbutils.fs.put(
    f"{STREAM_SOURCE}/drop_02.json",
    drop_02,
    overwrite=True
)

print("Drop 02 created successfully.")
display(dbutils.fs.ls(STREAM_SOURCE))

Wrote 779 bytes.
Drop 02 created successfully.


path,name,size,modificationTime
dbfs:/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json,drop_01.json,1040,1791042972000
dbfs:/Volumes/quickbite/default/raw_files/streaming_events/drop_02.json,drop_02.json,779,1791043244000


In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", QUERY_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print("Drop 02 processed successfully.")

Drop 02 processed successfully.


In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS total_events
    FROM {BRONZE_TABLE}
""").show()

+------------+
|total_events|
+------------+
|           7|
+------------+



In [0]:
display(
    spark.sql(f"""
        SELECT event_id, order_id, event_type, city, _source_file
        FROM {BRONZE_TABLE}
        ORDER BY event_timestamp
    """)
)

event_id,order_id,event_type,city,_source_file
EVT000001,QBO10001,ORDER_PLACED,Hyderabad,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000002,QBO10001,RESTAURANT_ACCEPTED,Hyderabad,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000003,QBO10001,RIDER_ASSIGNED,Hyderabad,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000004,QBO10002,ORDER_PLACED,Pune,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000005,QBO10002,RESTAURANT_ACCEPTED,Pune,/Volumes/quickbite/default/raw_files/streaming_events/drop_02.json
EVT000006,QBO10002,RIDER_ASSIGNED,Pune,/Volumes/quickbite/default/raw_files/streaming_events/drop_02.json
EVT000007,QBO10003,ORDER_PLACED,Bengaluru,/Volumes/quickbite/default/raw_files/streaming_events/drop_02.json


In [0]:
# QuickBite - Streaming Silver

SILVER_TABLE = "quickbite.default.silver_order_events"

silver_df = (
    spark.table(BRONZE_TABLE)
    .select(
        "event_id",
        "order_id",
        "event_timestamp",
        "event_type",
        "city",
        "restaurant_id",
        "rider_id",
        "order_value_inr",
        "payment_method",
        "source_system",
        "_ingested_at",
        "_source_file"
    )
    .withColumn("event_type", F.upper(F.trim(F.col("event_type"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("payment_method", F.upper(F.trim(F.col("payment_method"))))
    .dropDuplicates(["event_id"])
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_TABLE)

print("Streaming Silver table created successfully.")

Streaming Silver table created successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {SILVER_TABLE}
        ORDER BY event_timestamp
    """)
)

event_id,order_id,event_timestamp,event_type,city,restaurant_id,rider_id,order_value_inr,payment_method,source_system,_ingested_at,_source_file
EVT000001,QBO10001,2026-10-03T18:00:00.000Z,ORDER_PLACED,Hyderabad,QBR0308,null,425.5,UPI,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000002,QBO10001,2026-10-03T18:02:00.000Z,RESTAURANT_ACCEPTED,Hyderabad,QBR0308,null,425.5,UPI,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000003,QBO10001,2026-10-03T18:05:00.000Z,RIDER_ASSIGNED,Hyderabad,QBR0308,QBD01314,425.5,UPI,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000004,QBO10002,2026-10-03T18:06:00.000Z,ORDER_PLACED,Pune,QBR0511,null,315.75,CARD,QUICKBITE_LIVE,2026-10-03T15:58:43.006Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_01.json
EVT000005,QBO10002,2026-10-03T18:08:00.000Z,RESTAURANT_ACCEPTED,Pune,QBR0511,null,315.75,CARD,QUICKBITE_LIVE,2026-10-03T16:01:16.295Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_02.json
EVT000006,QBO10002,2026-10-03T18:11:00.000Z,RIDER_ASSIGNED,Pune,QBR0511,QBD02107,315.75,CARD,QUICKBITE_LIVE,2026-10-03T16:01:16.295Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_02.json
EVT000007,QBO10003,2026-10-03T18:12:00.000Z,ORDER_PLACED,Bengaluru,QBR0724,null,589.0,UPI,QUICKBITE_LIVE,2026-10-03T16:01:16.295Z,/Volumes/quickbite/default/raw_files/streaming_events/drop_02.json


In [0]:
# QuickBite - Gold Live Order Status

GOLD_TABLE = "quickbite.default.gold_live_order_status"

gold_df = spark.sql(f"""
    SELECT
        order_id,
        event_timestamp AS latest_event_timestamp,
        event_type AS current_status,
        city,
        restaurant_id,
        rider_id,
        order_value_inr,
        payment_method,
        source_system
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY order_id
                   ORDER BY event_timestamp DESC
               ) AS rn
        FROM {SILVER_TABLE}
    )
    WHERE rn = 1
""")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_TABLE)

print("Gold Live Order Status created successfully.")

Gold Live Order Status created successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_TABLE}
        ORDER BY order_id
    """)
)

order_id,latest_event_timestamp,current_status,city,restaurant_id,rider_id,order_value_inr,payment_method,source_system
QBO10001,2026-10-03T18:05:00.000Z,RIDER_ASSIGNED,Hyderabad,QBR0308,QBD01314,425.5,UPI,QUICKBITE_LIVE
QBO10002,2026-10-03T18:11:00.000Z,RIDER_ASSIGNED,Pune,QBR0511,QBD02107,315.75,CARD,QUICKBITE_LIVE
QBO10003,2026-10-03T18:12:00.000Z,ORDER_PLACED,Bengaluru,QBR0724,null,589.0,UPI,QUICKBITE_LIVE


In [0]:
# QuickBite - Streaming Data Quality Checks

dq_result = spark.sql(f"""
    SELECT
        COUNT(*) AS total_orders,
        COUNT(DISTINCT order_id) AS unique_orders,
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_ids,
        SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) AS null_event_types,
        SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS null_cities,
        SUM(CASE WHEN order_value_inr < 0 THEN 1 ELSE 0 END) AS negative_order_values
    FROM {SILVER_TABLE}
""")

display(dq_result)

total_orders,unique_orders,null_order_ids,null_event_types,null_cities,negative_order_values
7,3,0,0,0,0


In [0]:
# QuickBite - Streaming Gold KPIs

GOLD_KPI_TABLE = "quickbite.default.gold_live_kpis"

kpi_df = spark.sql(f"""
    SELECT
        COUNT(*) AS total_events,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(CASE WHEN event_type = 'ORDER_PLACED' THEN 1 ELSE 0 END) AS orders_placed,
        SUM(CASE WHEN event_type = 'RESTAURANT_ACCEPTED' THEN 1 ELSE 0 END) AS restaurants_accepted,
        SUM(CASE WHEN event_type = 'RIDER_ASSIGNED' THEN 1 ELSE 0 END) AS riders_assigned,
        ROUND(SUM(order_value_inr), 2) AS total_order_value_inr,
        ROUND(AVG(order_value_inr), 2) AS average_order_value_inr
    FROM {SILVER_TABLE}
""")

kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_KPI_TABLE)

print("Gold KPI table created successfully.")

Gold KPI table created successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_KPI_TABLE}
    """)
)

total_events,total_orders,orders_placed,restaurants_accepted,riders_assigned,total_order_value_inr,average_order_value_inr
7,3,3,2,2,2812.75,401.82


In [0]:
# Correct order-level Gold KPIs

GOLD_KPI_TABLE = "quickbite.default.gold_live_kpis"

kpi_df = spark.sql(f"""
    SELECT
        COUNT(*) AS total_orders,

        SUM(CASE
            WHEN current_status = 'ORDER_PLACED'
            THEN 1 ELSE 0
        END) AS orders_placed,

        SUM(CASE
            WHEN current_status = 'RESTAURANT_ACCEPTED'
            THEN 1 ELSE 0
        END) AS restaurants_accepted,

        SUM(CASE
            WHEN current_status = 'RIDER_ASSIGNED'
            THEN 1 ELSE 0
        END) AS riders_assigned,

        ROUND(SUM(order_value_inr), 2) AS total_order_value_inr,

        ROUND(AVG(order_value_inr), 2) AS average_order_value_inr

    FROM {GOLD_TABLE}
""")

kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_KPI_TABLE)

print("Corrected Gold KPI table created successfully.")

Corrected Gold KPI table created successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_KPI_TABLE}
    """)
)

total_events,total_orders,orders_placed,restaurants_accepted,riders_assigned,total_order_value_inr,average_order_value_inr
null,3,1,0,2,1330.25,443.42


In [0]:
# Final Gold KPI table

GOLD_KPI_TABLE = "quickbite.default.gold_live_kpis"

kpi_df = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_TABLE}) AS total_events,

        COUNT(*) AS total_orders,

        SUM(CASE
            WHEN current_status = 'ORDER_PLACED'
            THEN 1 ELSE 0
        END) AS orders_placed,

        SUM(CASE
            WHEN current_status = 'RESTAURANT_ACCEPTED'
            THEN 1 ELSE 0
        END) AS restaurants_accepted,

        SUM(CASE
            WHEN current_status = 'RIDER_ASSIGNED'
            THEN 1 ELSE 0
        END) AS riders_assigned,

        ROUND(SUM(order_value_inr), 2) AS total_order_value_inr,

        ROUND(AVG(order_value_inr), 2) AS average_order_value_inr

    FROM {GOLD_TABLE}
""")

kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_KPI_TABLE)

print("Final Gold KPI table created successfully.")

Final Gold KPI table created successfully.


In [0]:
# QuickBite - Incremental Streaming

incremental_query = (
    stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", QUERY_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

incremental_query.awaitTermination()

print("Incremental streaming completed successfully.")

Incremental streaming completed successfully.


In [0]:
drop_03 = """
{"event_id":"EVT000008","order_id":"QBO10003","event_timestamp":"2026-10-03T18:15:00","event_type":"RESTAURANT_ACCEPTED","city":"Bengaluru","restaurant_id":"QBR0724","rider_id":null,"order_value_inr":589.00,"payment_method":"UPI","source_system":"QUICKBITE_LIVE"}
{"event_id":"EVT000009","order_id":"QBO10003","event_timestamp":"2026-10-03T18:18:00","event_type":"RIDER_ASSIGNED","city":"Bengaluru","restaurant_id":"QBR0724","rider_id":"QBD03112","order_value_inr":589.00,"payment_method":"UPI","source_system":"QUICKBITE_LIVE"}
{"event_id":"EVT000010","order_id":"QBO10004","event_timestamp":"2026-10-03T18:19:00","event_type":"ORDER_PLACED","city":"Hyderabad","restaurant_id":"QBR0142","rider_id":null,"order_value_inr":275.00,"payment_method":"UPI","source_system":"QUICKBITE_LIVE"}
"""

dbutils.fs.put(
    f"{STREAM_SOURCE}/drop_03.json",
    drop_03,
    overwrite=True
)

print("Drop 03 created successfully.")

Wrote 787 bytes.
Drop 03 created successfully.


In [0]:
incremental_query = (
    stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", QUERY_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

incremental_query.awaitTermination()

print("Drop 03 processed successfully.")

Drop 03 processed successfully.


In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS total_events
    FROM {BRONZE_TABLE}
""").show()

+------------+
|total_events|
+------------+
|          10|
+------------+



In [0]:
# Refresh Silver from Streaming Bronze

silver_df = (
    spark.table(BRONZE_TABLE)
    .select(
        "event_id",
        "order_id",
        "event_timestamp",
        "event_type",
        "city",
        "restaurant_id",
        "rider_id",
        "order_value_inr",
        "payment_method",
        "source_system",
        "_ingested_at",
        "_source_file"
    )
    .withColumn("event_type", F.upper(F.trim(F.col("event_type"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("payment_method", F.upper(F.trim(F.col("payment_method"))))
    .dropDuplicates(["event_id"])
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print("Silver refreshed successfully.")

Silver refreshed successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT event_id, order_id, event_type, city
        FROM {SILVER_TABLE}
        ORDER BY event_timestamp
    """)
)

event_id,order_id,event_type,city
EVT000001,QBO10001,ORDER_PLACED,Hyderabad
EVT000002,QBO10001,RESTAURANT_ACCEPTED,Hyderabad
EVT000003,QBO10001,RIDER_ASSIGNED,Hyderabad
EVT000004,QBO10002,ORDER_PLACED,Pune
EVT000005,QBO10002,RESTAURANT_ACCEPTED,Pune
EVT000006,QBO10002,RIDER_ASSIGNED,Pune
EVT000007,QBO10003,ORDER_PLACED,Bengaluru
EVT000008,QBO10003,RESTAURANT_ACCEPTED,Bengaluru
EVT000009,QBO10003,RIDER_ASSIGNED,Bengaluru
EVT000010,QBO10004,ORDER_PLACED,Hyderabad


In [0]:
gold_df = spark.sql(f"""
    SELECT
        order_id,
        event_timestamp AS latest_event_timestamp,
        event_type AS current_status,
        city,
        restaurant_id,
        rider_id,
        order_value_inr,
        payment_method,
        source_system
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY order_id
                   ORDER BY event_timestamp DESC
               ) AS rn
        FROM {SILVER_TABLE}
    )
    WHERE rn = 1
""")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)

print("Gold Live Order Status refreshed successfully.")

Gold Live Order Status refreshed successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_TABLE}
        ORDER BY order_id
    """)
)

order_id,latest_event_timestamp,current_status,city,restaurant_id,rider_id,order_value_inr,payment_method,source_system
QBO10001,2026-10-03T18:05:00.000Z,RIDER_ASSIGNED,Hyderabad,QBR0308,QBD01314,425.5,UPI,QUICKBITE_LIVE
QBO10002,2026-10-03T18:11:00.000Z,RIDER_ASSIGNED,Pune,QBR0511,QBD02107,315.75,CARD,QUICKBITE_LIVE
QBO10003,2026-10-03T18:18:00.000Z,RIDER_ASSIGNED,Bengaluru,QBR0724,QBD03112,589.0,UPI,QUICKBITE_LIVE
QBO10004,2026-10-03T18:19:00.000Z,ORDER_PLACED,Hyderabad,QBR0142,null,275.0,UPI,QUICKBITE_LIVE


In [0]:
kpi_df = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_TABLE}) AS total_events,

        COUNT(*) AS total_orders,

        SUM(CASE
            WHEN current_status = 'ORDER_PLACED'
            THEN 1 ELSE 0
        END) AS orders_placed,

        SUM(CASE
            WHEN current_status = 'RESTAURANT_ACCEPTED'
            THEN 1 ELSE 0
        END) AS restaurants_accepted,

        SUM(CASE
            WHEN current_status = 'RIDER_ASSIGNED'
            THEN 1 ELSE 0
        END) AS riders_assigned,

        ROUND(SUM(order_value_inr), 2) AS total_order_value_inr,

        ROUND(AVG(order_value_inr), 2) AS average_order_value_inr

    FROM {GOLD_TABLE}
""")

kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_KPI_TABLE)

print("Gold KPIs refreshed successfully.")

Gold KPIs refreshed successfully.


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {GOLD_KPI_TABLE}
    """)
)

total_events,total_orders,orders_placed,restaurants_accepted,riders_assigned,total_order_value_inr,average_order_value_inr
10,4,1,0,3,1605.25,401.31


In [0]:
spark.sql("SHOW TABLES IN quickbite.default").show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|default |bronze_order_events   |false      |
|default |gold_live_kpis        |false      |
|default |gold_live_order_status|false      |
|default |silver_order_events   |false      |
+--------+----------------------+-----------+



In [0]:
spark.sql("DESCRIBE TABLE EXTENDED quickbite.default.silver_order_events").show(truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                 |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|event_id                    |string                                                                                                                                                    |NULL   |
|order_id                    |string                                                                                                                                                    |NULL   |
|event_timestamp             |

In [0]:
spark.sql("DESCRIBE TABLE EXTENDED quickbite.default.gold_live_order_status").show(truncate=False)

+----------------------------+-------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                      |comment|
+----------------------------+-------------------------------------------------------------------------------------------------------------------------------+-------+
|order_id                    |string                                                                                                                         |NULL   |
|latest_event_timestamp      |timestamp                                                                                                                      |NULL   |
|current_status              |string                                                                                                                         |NULL   

In [0]:
spark.sql("SHOW TABLES IN quickbite.default").show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|default |bronze_order_events   |false      |
|default |gold_live_kpis        |false      |
|default |gold_live_order_status|false      |
|default |silver_order_events   |false      |
+--------+----------------------+-----------+



In [0]:
display(
    spark.sql("""
        SELECT
            total_events,
            total_orders,
            orders_placed,
            restaurants_accepted,
            riders_assigned,
            total_order_value_inr,
            average_order_value_inr
        FROM quickbite.default.gold_live_kpis
    """)
)

total_events,total_orders,orders_placed,restaurants_accepted,riders_assigned,total_order_value_inr,average_order_value_inr
10,4,1,0,3,1605.25,401.31


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM quickbite.default.gold_live_kpis
    """)
)

total_events,total_orders,orders_placed,restaurants_accepted,riders_assigned,total_order_value_inr,average_order_value_inr
10,4,1,0,3,1605.25,401.31


In [0]:
display(
    spark.sql("""
        SELECT
            total_events,
            total_orders,
            orders_placed,
            restaurants_accepted,
            riders_assigned,
            total_order_value_inr,
            average_order_value_inr
        FROM quickbite.default.gold_live_kpis
    """)
)

total_events,total_orders,orders_placed,restaurants_accepted,riders_assigned,total_order_value_inr,average_order_value_inr
10,4,1,0,3,1605.25,401.31
